# starling — single-scan DFXM analysis (beamline notebook)

GPU-accelerated DFXM workflow for one scan. Work through the
cells top to bottom; each analysis cell is independent once the data is loaded.

The device (CUDA on the ESRF cluster, MPS on a Mac, CPU fallback) is detected
automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import starling

print("compute device:", starling.get_device(None))

## 1. Load a scan

Point at the BLISS master file and scan id.

In [ ]:
PATH = "/Volumes/LaCie/ESRF_MA6278/RAW_DATA/<sample>/<dataset>/<dataset>.h5"  # EDIT
SCAN_ID = "1.1"  # EDIT

dset = starling.DataSet(PATH, scan_id=SCAN_ID)
dset.info()
print("data shape:", dset.data.shape)

## 2. Background subtraction and ROI

In [ ]:
bg = dset.estimate_background(n_lowest=5, mode="mean")
dset.subtract(bg)

roi = dset.auto_roi(threshold_rel=0.05, pad=20)   # crops dset.data in place
print("auto ROI:", roi, "->", dset.data.shape)

plt.figure(figsize=(5, 5))
plt.imshow(dset.data.reshape(*dset.data.shape[:2], -1).sum(-1), cmap="magma")
plt.title("z-sum after background + ROI"); plt.colorbar(); plt.show()

## 3. Moments — COM and mosaicity maps

Per-pixel intensity-weighted mean and covariance.

In [ ]:
mean, cov = dset.moments()

if mean.ndim == 3:  # mosa scan: 2 motors
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    im0 = ax[0].imshow(mean[..., 0]); ax[0].set_title("COM motor 0"); plt.colorbar(im0, ax=ax[0])
    im1 = ax[1].imshow(mean[..., 1]); ax[1].set_title("COM motor 1"); plt.colorbar(im1, ax=ax[1])
    mosa = np.sqrt(cov[..., 0, 0] + cov[..., 1, 1])
    im2 = ax[2].imshow(mosa); ax[2].set_title("mosaicity (sqrt tr cov)"); plt.colorbar(im2, ax=ax[2])
else:               # rocking scan: 1 motor
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    im0 = ax[0].imshow(mean); ax[0].set_title("COM"); plt.colorbar(im0, ax=ax[0])
    im1 = ax[1].imshow(np.sqrt(cov)); ax[1].set_title("width (sqrt var)"); plt.colorbar(im1, ax=ax[1])
plt.show()

## 4. Rocking-curve fitting (1D scans)

Batched Gauss-Newton on the GPU. Output:
`[A, sigma, mu, k, m, success]` per pixel.

In [ ]:
if dset.data.ndim == 3:
    params = dset.fit_1D_gaussian()
    ok = params[..., 5] > 0
    FWHM = 2 * np.sqrt(2 * np.log(2))
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    im0 = ax[0].imshow(np.where(ok, params[..., 2], np.nan)); ax[0].set_title("peak centre mu"); plt.colorbar(im0, ax=ax[0])
    im1 = ax[1].imshow(np.where(ok, params[..., 1] * FWHM, np.nan)); ax[1].set_title("FWHM"); plt.colorbar(im1, ax=ax[1])
    im2 = ax[2].imshow(np.where(ok, params[..., 0], np.nan)); ax[2].set_title("amplitude"); plt.colorbar(im2, ax=ax[2])
    plt.show()
    print(f"success rate: {ok.mean():.1%}")

## 5. Two-peak fit — bimodal / two-grain pixels

Fits 1-peak and 2-peak models, selects per pixel by BIC + physical gates.
`n_peaks == 2` marks pixels with two resolved peaks (e.g. overlapping grains).

In [ ]:
if dset.data.ndim == 3:
    out2 = dset.fit_two_gaussians_1D()
    n_peaks = out2["n_peaks"]
    print(f"2-peak pixels: {(n_peaks == 2).mean():.2%}")

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(n_peaks, vmin=0, vmax=2); ax[0].set_title("n_peaks (0/1/2)")
    p2 = out2["params2"]
    sep = np.where(n_peaks == 2, p2[..., 5] - p2[..., 2], np.nan)
    im = ax[1].imshow(sep); ax[1].set_title("peak separation (motor units)"); plt.colorbar(im, ax=ax[1])
    plt.show()

## 6. 2D Gaussian fit (mosa scans)

Sub-step orientation centres and the full 2x2 orientation covariance per
pixel: `[A, mu0, mu1, cov00, cov01, cov11, c, success]`.

In [ ]:
if dset.data.ndim == 4:
    p2d = dset.fit_2D_gaussian()
    ok = p2d[..., 7] > 0
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    im0 = ax[0].imshow(np.where(ok, p2d[..., 1], np.nan)); ax[0].set_title("centre motor 0"); plt.colorbar(im0, ax=ax[0])
    im1 = ax[1].imshow(np.where(ok, p2d[..., 2], np.nan)); ax[1].set_title("centre motor 1"); plt.colorbar(im1, ax=ax[1])
    aniso = np.sqrt(np.where(ok, p2d[..., 3] / np.maximum(p2d[..., 5], 1e-12), np.nan))
    im2 = ax[2].imshow(aniso); ax[2].set_title("width anisotropy"); plt.colorbar(im2, ax=ax[2])
    plt.show()

## 7. Save results

In [ ]:
dset.save_maps("scan_results.h5", {
    "mean": mean,
    "covariance": cov,
    # add fit outputs you computed above, e.g.:
    # "gauss1d": {"params": params},
    # "gauss2p": out2,
})
print("saved scan_results.h5")